# 3D Toric-Code NQS — hyperparameter sweep on Colab

Runs the **same** `Three_TC.train` pipeline as the NERSC GPU job array. On Colab
you run **each config as its own cell** — one fresh `!python` process per cell,
which is the pattern that runs reliably here (the automated loop could skip runs).

**Run cells 1–5 top to bottom once** (setup), then use the **run cells** at the
bottom: copy one, change `NAME` and the training hyperparameters, and run it.

The network is the **code default** in `Three_TC/model/networks.py` — non-invariant
`[4, 4]`, invariant `[4, 4, 1]` (~2.6k params). Each run does dense-QGT Stochastic
Reconfiguration and prints, every step, `E ± ΔE`, the spread `√Var`, and the figure
of merit **`delta = |E − E_exact| / |E_exact|`** (also logged to W&B).

> First: **Runtime → Change runtime type → GPU**, then run cell 1 to confirm.

**Boundary conditions.** Set `BC = "PBC"` or `"OBC"` in the configure cell. PBC uses the hardcoded `--hz_preset` ED reference; OBC (L=2, N=12) diagonalizes the exact ground state **on the fly** and passes it as `--exact_E0`, so `delta` works for both. OBC runs land in their own `outputs/colab-obc-…` folder.

In [ ]:
# 1. Confirm a GPU is attached (Runtime -> Change runtime type -> GPU).
!nvidia-smi -L || echo "NO GPU -- switch the runtime to GPU before continuing."

In [ ]:
# 2. Clone (or update) the repo into the Colab VM. Idempotent: safe to re-run.
import os
REPO_URL = "https://github.com/SanzharBissenali/ThreeD_TC.git"
REPO_DIR = "/content/ThreeD_TC"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git pull --ff-only || true
print("cwd:", os.getcwd())

In [ ]:
# 3. Install the NQS stack, pinned to match the NERSC env (parity of results).
#    Colab may print dependency-conflict warnings for unrelated preinstalled
#    packages (tensorflow, etc.) -- those are harmless for this pipeline.
!pip install -q jax==0.5.2 jaxlib==0.5.1 \
    jax-cuda12-plugin==0.5.1 jax-cuda12-pjrt==0.5.1 \
    netket==3.16.1.post1 flax==0.10.4 optax numba wandb tqdm

In [ ]:
# 4. GATE: make sure JAX actually sees the GPU (expect platform 'gpu').
#    If this fails: re-run cell 3, then Runtime -> Restart session, then re-run
#    from cell 2. (x64 is already enabled inside Three_TC/train.py.)
import os
# Allocate GPU memory on demand instead of grabbing ~75% up front -- keeps
# back-to-back runs from fighting over VRAM. Set BEFORE importing jax.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
import jax
print("devices:", jax.devices(), "| backend:", jax.default_backend())
assert jax.default_backend() == "gpu", "JAX is not on the GPU -- see the note above."

## Weights & Biases

`WANDB = True` logs every step's `delta` / `energy` / `energy_abs_err` to a shared
group so all configs plot side by side (a parallel-coordinates plot over `delta` is
the intended view). Use a group name **distinct from your NERSC group** so the two
don't mix — same project, so they're still visible together.

Set `WANDB = False` to skip it entirely and just watch the per-step stdout.

In [ ]:
# 5. W&B auth. Set WANDB = False to skip and rely on stdout only.
WANDB = True
if WANDB:
    import wandb
    wandb.login()   # paste your API key when prompted

## Configure — fixed args + a training menu  ← edit here

Set the shared args, the architecture, and the per-run output folder once; every run
cell below picks them up. `ARCH_FLAGS = ""` uses the **code default** net
(non-invariant `[4,4]`, invariant `[4,4,1]`, ~2.6k params) — set it to override. The
printed menu lists **training**-hyperparameter combos to copy into a run cell.

Pick `BC` (`PBC`/`OBC`) here as well: OBC computes its exact `E0` on the fly (cheap at L=2) and writes to a separate `colab-obc-…` folder, while PBC is unchanged.

In [ ]:
# === EDIT: fixed args shared by every run ==================================
BC        = "PBC"      # boundary conditions: "PBC" | "OBC"
L         = 2
HX        = 0.2        # transverse field (keep 0.2 for PBC: the presets are at hx=0.2)
HZ        = 0.2        # OBC only: the hz value (PBC takes hz from HZ_PRESET below)
HZ_PRESET = "easy"     # PBC ED reference point: hard | mid | easy  (ignored for OBC)
N_ITER    = 250
N_SAMPLES = 16384
N_CHAINS  = 1024
N_SWEEPS  = 48         # Metropolis sweeps between samples (code default 2N = 48 at L=2)

# Architecture: ARCH_FLAGS="" uses the CODE DEFAULT (Three_TC/model/networks.py):
#   non-invariant [4, 4]    (noninv_channels=4, n_noninv=2)
#   invariant     [4, 4, 1] (inv_hidden=(4,4) + final width-1)   -> ~2.6k params
# To try a different net, e.g.:
#   ARCH_FLAGS = "--noninv_channels 8 --n_noninv 1 --inv_hidden 8"
ARCH_FLAGS = ""
ARCH_TAG   = "cnn-4-4_4-4-1"          # label -> output folder + wandb group
# ===========================================================================

# Field / exact-E0 spec. PBC uses the hardcoded --hz_preset reference (hx=0.2).
# OBC has no preset, so we diagonalize the L=2 OBC Hamiltonian ON THE FLY (N=12,
# 2^12 states -> a fraction of a second) with the same verified ED as the repo,
# and pass E0 to --exact_E0 so the `delta` figure of merit works for OBC too.
if BC == "OBC":
    import numpy as np
    from scipy.sparse.linalg import eigsh
    from Three_TC.model.geometry import ThreeD_ToricCodeGeometry
    from model.exact_diag import hamiltonian_linop
    _geo = ThreeD_ToricCodeGeometry(L, L, L, bc="OBC")
    _H, _ = hamiltonian_linop(_geo, hx=HX, hz=HZ, J=1.0)
    E0_EXACT = float(eigsh(_H, k=1, which="SA", return_eigenvectors=False)[0])
    FIELD_FLAGS = f"--hx {HX} --hz {HZ} --exact_E0 {E0_EXACT}"
    POINT_TAG   = f"hx{HX}_hz{HZ}"
    print(f"OBC L={L} on-the-fly ED:  E0_exact = {E0_EXACT:.6f}  (N={_geo.N})")
else:
    FIELD_FLAGS = f"--hx {HX} --hz_preset {HZ_PRESET}"
    POINT_TAG   = HZ_PRESET

WANDB_GROUP = f"colab-{BC.lower()}-{POINT_TAG}-{ARCH_TAG}"
OUT_DIR     = f"outputs/{WANDB_GROUP}"   # fresh per-(bc, point, arch) folder

# Reference menu of TRAINING hyperparameters to try (copy a line into a run cell):
DT_LRMIN   = [(0.02, 0.002), (0.01, 0.001)]   # (initial lr, final lr)
DIAG_SHIFT = [1e-4, 1e-3, 1e-2]               # SR / QGT regularization

print(f"bc: {BC}   field flags: {FIELD_FLAGS}")
print("arch:", "code default [4,4]/[4,4,1]" if not ARCH_FLAGS else ARCH_FLAGS)
print(f"output folder: {OUT_DIR}/   |   "
      + (f"wandb group: {WANDB_GROUP}" if WANDB else "wandb OFF"))
print("training-hyperparameter menu (copy a line into a run cell):")
for (dt, lrm) in DT_LRMIN:
    for ds in DIAG_SHIFT:
        print(f"  --dt {dt} --lr_min {lrm} --diag_shift {ds}")

## Sanity check — one short run

A quick low-cost run to confirm training works end-to-end with the default net. The
`[train]` line should report `n_params=2571` (the `[4,4]/[4,4,1]` net), and you
should see the per-step `delta = ...` stream.

In [ ]:
# 6. One short smoke run (uses ARCH_FLAGS = the default [4,4]/[4,4,1] net).
wb = f"--wandb_group {WANDB_GROUP}" if WANDB else "--no_wandb"
!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_full \
  {FIELD_FLAGS} --n_iter 50 --n_samples 4096 --n_chains 256 \
  --n_sweeps {N_SWEEPS} --qgt dense --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {ARCH_FLAGS} --out_dir {OUT_DIR} --name cfg_smoke {wb}

## Run configs — one cell per run

**This is the reliable path:** each cell is a single fresh `!python` process (no
loop). To run a config: **copy a cell below, change `NAME`, edit the training flags
(`--dt --lr_min --diag_shift`), and run it.** The network comes from `ARCH_FLAGS`
(the default `[4,4]/[4,4,1]` unless you overrode it in the configure cell). Each
writes `OUT_DIR/<NAME>.{json,mpack}` and logs to `WANDB_GROUP`; `--qgt dense` keeps
the SR solve exact and GPU-safe.

> Change `NAME` every run so outputs/W&B don't overwrite each other.

In [ ]:
# RUN A - copy this whole cell per config; edit NAME + the training flags.
NAME = "cfg_a"
wb = f"--wandb_group {WANDB_GROUP}" if WANDB else "--no_wandb"
!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_full \
  {FIELD_FLAGS} --n_iter {N_ITER} --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {ARCH_FLAGS} --out_dir {OUT_DIR} --name {NAME} {wb}

In [ ]:
# RUN B - same pattern, a different NAME + training hyperparameters.
NAME = "cfg_b"
wb = f"--wandb_group {WANDB_GROUP}" if WANDB else "--no_wandb"
!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_full \
  {FIELD_FLAGS} --n_iter {N_ITER} --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.01 --lr_min 0.001 --diag_shift 1e-3 \
  {ARCH_FLAGS} --out_dir {OUT_DIR} --name {NAME} {wb}

## Notes

**Vary hyperparameters:** copy a run cell, set `NAME`, and edit the training flags
(`--dt --lr_min --diag_shift`). To change the **network**, set `ARCH_FLAGS` once in
the configure cell (e.g. `--noninv_channels 8 --n_noninv 1 --inv_hidden 8`) and bump
`ARCH_TAG` so the new net's runs land in a fresh folder. Change the physical point
via `HZ_PRESET`.

**Why one cell per run:** each `!python` call is an isolated fresh process, which is
what runs reliably on Colab; the automated loop could skip runs, so it's unrolled.

**Outputs.** Each run writes `OUT_DIR/<NAME>.json` (config + curve + final
observables) and `OUT_DIR/<NAME>.mpack` (weights) — a fresh per-arch folder
(`outputs/colab-<preset>-<arch_tag>/`), so runs don't pile up with previous ones.
The Colab VM is ephemeral — download keepers with the last cell, or rely on W&B.

**Boundary conditions.** `BC = "OBC"` switches every run to open boundaries via `--bc OBC` and diagonalizes the L=2 OBC ground state on the fly for the `delta` FOM (no preset needed); PBC keeps using `--hz_preset`. `BC` is folded into `WANDB_GROUP`/`OUT_DIR`, so OBC and PBC runs never share a folder. Only the symmetry-aware `ToricCNN`/`ToricCNN_full` support OBC (the Vanilla nets are PBC-only).

In [ ]:
# Optional: zip + download THIS run folder before the VM recycles.
from google.colab import files
!zip -qr /content/{WANDB_GROUP}.zip {OUT_DIR} && echo "zipped {OUT_DIR}"
files.download(f"/content/{WANDB_GROUP}.zip")